# 1 El Workflow estándar base para el desarrollo de proyectos de Machine Learning
## El Workflow de referencia por medio de Scikit-Learn

### Escenario de Aprendizaje: El Caso de MedTelecomunicaciones

**Tu Rol:** Has sido contratado(a) como Científico(a) de Datos Junior por la compañía de telecomunicaciones **MedTelecomunicaciones**.

**La Situación:**
En la reunión directiva de inicio de mes, el Gerente de Operaciones presentó un informe preocupante, la tasa de cancelación de clientes (*Churn*) ha aumentado un 18% en el último año. Adquirir un cliente nuevo le cuesta a la empresa aproximadamente 300 USD, mientras que implementar una campaña de fidelización para retener a un cliente existente cuesta solo 40 USD.

La junta directiva te ha asignado el proyecto de Machine Learning con el siguiente encargo oficial:

> *"Necesitamos que analices la base histórica de clientes y construyas un modelo predictivo capaz de identificar con precisión qué clientes están en riesgo crítico de irse el próximo mes. Queremos usar tus predicciones para que el equipo de fidelización ejecute campañas preventivas antes de que los clientes cancelen sus contratos."*

**Tu Misión "si decides aceptarla ;)":**
Desarrollar el flujo de referencia paso a paso, desde la comprensión del problema y el diagnóstico de los datos, hasta el entrenamiento de un modelo base con Scikit-Learn, documentando cada decisión técnica con criterio de negocio.

Nota: La medida para dinero son USD, en concordancia con los valores del conjunto de datos.
---

### Buena Práctica 1 Comprensión del Problema y Objetivos de Negocio

Debemos traducir una necesidad en una solución predictiva.

* **El Contexto de Negocio:** La tasa de cancelación o abandono (*Customer Churn*) impacta directamente los ingresos en la industria de telecomunicaciones. Capturar un cliente nuevo cuesta entre 5 y 25 veces más que retener a uno existente.
* **El Objetivo Predictivo:** Desarrollar un flujo de trabajo capaz de identificar qué clientes están en riesgo de deserción para anticipar acciones de fidelización.

In [ ]:
import pandas as pd

In [ ]:
# Repositorio del conjunto de datos Telco Customer Churn, elaborado por IBM
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

In [ ]:
# Carga del DataFrame
churn_df = pd.read_csv(DATA_URL)

In [ ]:
# Inspección básica de dimensiones y tipo de datos
print(f"Dimensiones del conjunto de datos: {churn_df.shape[0]} filas x {churn_df.shape[1]} columnas\n")
churn_df.info()

Dimensiones del conjunto de datos: 7043 filas x 21 columnas

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-n

# 📌 Explicación Conceptual: ¿Por qué usamos `[0]` en `churn_df.shape[0]`?

En **Pandas**, la propiedad `.shape` entrega la dimensión del conjunto de datos mediante una **tupla** ordenada con la estructura `(filas, columnas)`.

Dado que en Python la indexación comienza siempre desde cero (**indexación basada en cero**), se accede a cada dimensión mediante su posición:

---

### 💡 Esquema Conceptual

* **`churn_df.shape`** $\rightarrow$ `(número_de_filas, número_de_columnas)`
* **`churn_df.shape[0]`** $\rightarrow$ Accede a la **posición 0** (primer elemento): **Número total de filas**.
* **`churn_df.shape[1]`** $\rightarrow$ Accede a la **posición 1** (segundo elemento): **Número total de columnas**.



In [ ]:
# 1. Inspeccionar la tupla completa
dimensiones = churn_df.shape
print(f"Objeto shape: {dimensiones}")
print(f"Tipo de dato: {type(dimensiones)}")

print("-" * 40)

# 2. Acceder por índice a la tupla (0 y 1)
num_filas = churn_df.shape[0]
num_columnas = churn_df.shape[1]

print(f"Índice [0] -> Filas:    {num_filas}")
print(f"Índice [1] -> Columnas: {num_columnas}")

print("-" * 40)

Objeto shape: (7043, 21)
Tipo de dato: <class 'tuple'>
----------------------------------------
Índice [0] -> Filas:    7043
Índice [1] -> Columnas: 21
----------------------------------------


In [ ]:
churn_df.head(10).T

,0,1,2,3,4,5,6,7,8,9
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU,9305-CDSKC,1452-KIOVK,6713-OKOMC,7892-POOKP,6388-TABGU
gender,Female,Male,Male,Male,Female,Female,Male,Female,Female,Male
SeniorCitizen,0,0,0,0,0,0,0,0,0,0
Partner,Yes,No,No,No,No,No,No,No,Yes,No
Dependents,No,No,No,No,No,No,Yes,No,No,Yes
tenure,1,34,2,45,2,8,22,10,28,62
PhoneService,No,Yes,Yes,No,Yes,Yes,Yes,No,Yes,Yes
MultipleLines,No phone service,No,No,No phone service,No,Yes,Yes,No phone service,Yes,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic,Fiber optic,Fiber optic,DSL,Fiber optic,DSL
OnlineSecurity,No,Yes,Yes,Yes,No,No,No,Yes,No,Yes


### Buena Práctica 2: Exploración y Comprensión del Conjunto de Datos

Una vez conocemos el objetivo de negocio, debemos así mismo, conocer las variables que conforman el conjunto de datos, cómo podemos organizarlas y entender así su significado:

#### 1. Identificación y Demografía del Cliente
* **`customerID`**: Identificador único del cliente.
* **`gender`**: Género del cliente (`Male`, `Female`).
* **`SeniorCitizen`**: Indica si el cliente es adulto mayor (`1`: Sí, `0`: No).
* **`Partner`**: Indica si el cliente tiene pareja (`Yes`, `No`).
* **`Dependents`**: Indica si el cliente tiene personas a cargo (`Yes`, `No`).

#### 2. Servicios Contratados
* **`tenure`**: Número de meses que el cliente ha permanecido en la compañía.
* **`PhoneService`**: Indica si tiene servicio telefónico (`Yes`, `No`).
* **`MultipleLines`**: Indica si tiene múltiples líneas telefónicas (`Yes`, `No`, `No phone service`).
* **`InternetService`**: Tipo de proveedor de internet (`DSL`, `Fiber optic`, `No`).
* **`OnlineSecurity`**: Servicio de seguridad en línea (`Yes`, `No`, `No internet service`).
* **`OnlineBackup`**: Servicio de respaldo en línea (`Yes`, `No`, `No internet service`).
* **`DeviceProtection`**: Servicio de protección de dispositivos (`Yes`, `No`, `No internet service`).
* **`TechSupport`**: Servicio de soporte técnico prioritario (`Yes`, `No`, `No internet service`).
* **`StreamingTV`**: Servicio de transmisión de TV (`Yes`, `No`, `No internet service`).
* **`StreamingMovies`**: Servicio de transmisión de películas (`Yes`, `No`, `No internet service`).

#### 3. Información del Contrato y Facturación
* **`Contract`**: Tipo de contrato del cliente (`Month-to-month`, `One year`, `Two year`).
* **`PaperlessBilling`**: Facturación electrónica sin papel (`Yes`, `No`).
* **`PaymentMethod`**: Método de pago (`Electronic check`, `Mailed check`, `Bank transfer (automatic)`, `Credit card (automatic)`).
* **`MonthlyCharges`**: Monto facturado mensualmente al cliente (Numérica continua).
* **`TotalCharges`**: Monto total facturado acumulado durante su permanencia (Numérica continua, almacenada como texto).

#### 4. Variable Objetivo (Target)
* **`Churn`**: Indica si el cliente canceló su servicio en el último mes (`Yes`, `No`).

In [ ]:
# Lista de valores que representan nulos implícitos en texto o número
valores_centinela = ['Unknown', 'unknown', 'N/A', '?', 'None', ' ', -999, 9999]

# Conteo de ocurrencias por columna
nulos_implicitos = churn_df.isin(valores_centinela).sum()
nulos_implicitos[nulos_implicitos > 0]

,0
TotalCharges,11


Este bloque de código realiza una **auditoría automatizada de nulos implícitos** sobre todo el DataFrame (`churn_df`). Su objetivo es rastrear y contar aquellas celdas que contienen "valores centinela" (etiquetas o números de relleno) que ocultan datos ausentes sin ser un `NaN` oficial de Python.

---

### 1. Definición de la lista de inspección (`valores_centinela`)

```python
valores_centinela = ['Unknown', 'unknown', 'N/A', '?', 'None', ' ', -999, 9999]

```

* **Qué hace:** Crea una lista en Python que actúa como un "filtro de búsqueda" o catálogo de patrones sospechosos.
* **Detalle técnico:** Mezcla tipos de datos (`str` e `int`) porque las fuentes de datos reales sufren inconsistencias de captura:
* **Variaciones de texto:** `'Unknown'`, `'unknown'`, `'N/A'`, `'?'`, `'None'`.
* **Espacios invisibles:** `' '` (un espacio en blanco común en campos de texto, como el que vimos en `TotalCharges`).
* **Códigos numéricos fuera de rango:** `-999`, `9999` (usados históricamente en bases de datos relacionales o encuestas para codificar la opción "No responde" o "Dato no disponible").



---

### 2. Detección y suma matricial (`churn_df.isin(...).sum()`)

```python
nulos_implicitos = churn_df.isin(valores_centinela).sum()

```

Este paso ejecuta dos operaciones en cadena:

1. **`churn_df.isin(valores_centinela)`:**
Evalúa **celda por celda** todo el DataFrame. Retorna una matriz booleana (*DataFrame de `True` y `False*`) del mismo tamaño exacto que `churn_df`:
* Retorna `True` si el valor de la celda coincide con **cualquiera** de los elementos de la lista `valores_centinela`.
* Retorna `False` si la celda contiene un valor válido o un `NaN` regular.


2. **`.sum()`:**
En Python, `True` equivale a $1$ y `False` equivale a $0$. Al sumar por defecto sobre el eje vertical (columnas), Pandas calcula el total de coincidencias encontradas por cada variable, devolviendo una Serie con el nombre de la columna y la cantidad de nulos implícitos.

---

### 3. Filtrado de resultados relevantes (`nulos_implicitos[...]`)

```python
nulos_implicitos[nulos_implicitos > 0]

```

* **Qué hace:** Aplica una máscara booleana sobre la Serie calculada en el paso anterior.
* **Detalle técnico:** Si el dataset tiene 20 o 30 columnas, la mayoría devolverá un conteo de $0$. Esta línea **filtra la salida para mostrar únicamente las columnas donde se detectó al menos un nulo implícito** ($> 0$), evitando el ruido en la pantalla del usuario o estudiante.

---

### Ejemplo conceptual de la transformación interna

Imaginemos una tabla simplificada de 3 filas y 2 columnas:

|  | `Education` | `MonthlyCharges` |
| --- | --- | --- |
| **Fila 0** | `'College'` | `70.5` |
| **Fila 1** | `'Unknown'` | `-999` |
| **Fila 2** | `'Doctorate'` | `20.0` |

#### Paso A: `churn_df.isin(['Unknown', -999])`

|  | `Education` | `MonthlyCharges` |
| --- | --- | --- |
| **Fila 0** | `False` ($0$) | `False` ($0$) |
| **Fila 1** | `True` ($1$) | `True` ($1$) |
| **Fila 2** | `False` ($0$) | `False` ($0$) |

#### Paso B: `.sum()`

```text
Education         1
MonthlyCharges    1
dtype: int64

```

#### Paso C: `[nulos_implicitos > 0]`

Muestra ambas columnas porque ambas superan la condición de tener más de 0 inconsistencias detectadas.

---

### Utilidad pedagógica

Esta técnica enseña a los estudiantes a **vectorizar la exploración de datos**, evitando escribir un bucle `for` manual o revisar columna por columna con `.value_counts()`, automatizando la detección de anomalías en una sola celda de ejecución.

In [ ]:
for col in churn_df.select_dtypes(include=['object']).columns:
    print(f"=== Variable: {col} ===")
    print(churn_df[col].value_counts(dropna=False))
    print("\n")

=== Variable: customerID ===
customerID
3186-AJIEK    1
7590-VHVEG    1
5575-GNVDE    1
8775-CEBBJ    1
2823-LKABH    1
             ..
6713-OKOMC    1
1452-KIOVK    1
9305-CDSKC    1
9237-HQITU    1
7795-CFOCW    1
Name: count, Length: 7043, dtype: int64


=== Variable: gender ===
gender
Male      3555
Female    3488
Name: count, dtype: int64


=== Variable: Partner ===
Partner
No     3641
Yes    3402
Name: count, dtype: int64


=== Variable: Dependents ===
Dependents
No     4933
Yes    2110
Name: count, dtype: int64


=== Variable: PhoneService ===
PhoneService
Yes    6361
No      682
Name: count, dtype: int64


=== Variable: MultipleLines ===
MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64


=== Variable: InternetService ===
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64


=== Variable: OnlineSecurity ===
OnlineSecurity
No                     3498
Yes            

Este bloque de código realiza una **inspección categórica automatizada** sobre todo el DataFrame (`churn_df`). Su objetivo es recorre columna por columna únicamente aquellas variables de tipo texto o categoría, mostrando la distribución de sus frecuencias y detectando valores nulos (tanto explícitos como implícitos).

---

### 1. La selección de columnas (`churn_df.select_dtypes(...)`)

```python
churn_df.select_dtypes(include=['object', 'category']).columns

```

* **`select_dtypes(include=['object', 'category'])`:** Es un filtro inteligente de Pandas. En lugar de evaluar todas las columnas del dataset (que podrían incluir variables numéricas como edad o cargos mensuales), le ordena a Pandas: *"Selecciona únicamente las columnas cuyo tipo de dato sea texto (`object`) o categoría explícita (`category`)"*.
* **`.columns`:** Extrae una lista/índice con únicamente los **nombres de esas columnas seleccionadas**. Por ejemplo: `['gender', 'Partner', 'Dependents', 'InternetService', ...]`.

---

### 2. La estructura del bucle (`for col in ...:`)

```python
for col in churn_df.select_dtypes(...).columns:

```

* Es un bucle `for` estándar en Python que **itera una por una** sobre las variables categóricas filtradas.
* En cada vuelta de la iteración, la variable `col` toma el nombre de una columna específica (por ejemplo, en la primera vuelta `col = 'gender'`, en la segunda `col = 'Partner'`, etc.).

---

### 3. El cuerpo del bucle (Impresión y Análisis)

```python
print(f"=== Variable: {col} ===")
print(churn_df[col].value_counts(dropna=False))
print("\n")

```

Cada iteración ejecuta tres acciones dentro de la consola:

1. **`print(f"=== Variable: {col} ===")`)**: Imprime un encabezado formateado (usando un *f-string*) para separar visualmente los resultados e identificar con claridad qué columna se está analizando.
2. **`churn_df[col].value_counts(dropna=False)`**:
* **`value_counts()`**: Cuenta cuántas veces se repite cada valor único dentro de esa columna específica.
* **`dropna=False` (Parámetro crítico)**: Por defecto, `.value_counts()` ignora los valores nulos (`NaN`). Al configurarlo en `False`, le obligamos a **incluir los datos faltantes explícitos** dentro del recuento si existieran, mostrándolos al final como `NaN: <cantidad>`.


3. **`print("\n")`**: Añade un salto de línea en blanco para dejar espacio visual entre el resultado de una variable y la siguiente, facilitando la lectura del reporte.

---

### Ejemplo de lo que genera en pantalla

Si el dataset tuviera las columnas `InternetService` y `Contract`, la salida generada sería:

```text
=== Variable: InternetService ===
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64


=== Variable: Contract ===
Contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64

```

---

### Utilidad didáctica para la clase

A diferencia de la búsqueda con `valores_centinela` (que requiere conocer previamente qué etiquetas buscar), este bucle le enseña al estudiante a **auditar el "vocabulario" completo de sus variables cualitativas**. Permite descubrir instantáneamente:

* Si hay etiquetas mal escritas o inconsistentes (ej. `'Yes'`, `'yes'`, `'YES'`).
* La presencia de categorías redundantes o raras (ej. `'No internet service'`).
* El desbalance de categorías en variables cualitativas clave.

### Buena Práctica 3: Identificación de Variables y Diagnóstico de Transformación

A partir de conocer las variables con las cuales vamos a trabajar, debemos clasificarlas según su naturaleza y determinar **qué ajuste se necesita y por qué**.

#### 1. Clasificación acorde a su tipología

* **Variables Numéricas (Cuantitativas):**
  * `tenure`: Antigüedad en meses (Discreta).
  * `MonthlyCharges`: Cargo mensual en dinero (Continua).
  * `TotalCharges`: Cargo total acumulado en dinero (Continua). *Nota: Actualmente almacenada como texto debido a registros con espacios en blanco.*

* **Variables Categóricas (Cualitativas):**
  * `gender`, `SeniorCitizen`, `Partner`, `Dependents`: Demográficas.
  * `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`: Servicios contratados.
  * `Contract`, `PaperlessBilling`, `PaymentMethod`: Contrato y pago.
  * `Churn`: Variable objetivo (Target).

---

#### 2. Diagnóstico de Necesidades de Cambio
*Nota: esto es un por qué

| Grupo de Variables | Tipo de Variable | Necesidad de Cambio | Justificación (¿Por qué?) |
| :--- | :--- | :--- | :--- |
| **`customerID`** | Texto (ID único) | **Eliminar del análisis** | Es un identificador administrativo. No representa un patrón de comportamiento y haría que el modelo memorice casos individuales en lugar de aprender. |
| **`TotalCharges`** | Numérica (almacenada como texto) | **Convertir a número y corregir vacíos** | Es una variable cuantitativa, pero la presencia de espacios en blanco hace que se interprete como texto. Para los clientes nuevos ($0$ meses de antigüedad), el valor real acumulado debe ser $0.0$. |
| **`Churn`** | Categórica Binaria (Target) | **Representar como $1$ y $0$** | La variable objetivo viene en texto (`Yes` / `No`). Los modelos matemáticos requieren que la presencia del evento de interés (*Churn*) se represente de forma binaria ($1$ para `Yes`, $0$ para `No`). |
| **Variables Categóricas** | Categóricas Nominales | **Convertir a representación numérica sin orden** | Las variables cualitativas no tienen un orden implícito (ej. tipo de internet o método de pago). Deben transformarse a formato numérico evitando crear jerarquías falsas ($1 < 2 < 3$) que confundan al algoritmo. |
| **Variables Numéricas** | Numéricas Continuas | **Llevar a una escala común** | Tienen rangos e intervalos muy dispares (ej. `MonthlyCharges` entre $18$ y $118$ vs. `TotalCharges` entre $0$ y $8600$). Si no se unifican las escalas, las variables con números más grandes tendrán un peso desproporcionado e injustificado en el modelo. |

### Buena Práctica 4 Procedimientos de Validación y Diagnóstico de la Calidad de Datos

En los proyectos relacionados con el Machine Learning, es muy importante, validar el conjunto de datos para garantizar que no existan inconsistencias que comprometan la estabilidad, el entrenamiento o el rendimiento de los algoritmos.

---

#### 1. Aspectos Mínimos a Evaluar

* **Identificación de Datos Faltantes (Nulos Explícitos e Implícitos):**
  * **Nulos Explícitos:** Valores ausentes reconocidos automáticamente por el entorno (representados como `NaN` o `None`).
  * **Nulos Implícitos:** Datos faltantes "enmascarados" como información válida. Ocurren cuando un campo vacío se llena con espacios en blanco (`" "`), textos como `"N/A"`, `"null"` o guiones (`"-"`). Esto distorsiona la interpretación del tipo de dato (por ejemplo, haciendo que una variable numérica sea interpretada como texto por Pandas).
* **Consistencia de Formatos y Unificación de Valores:**
  * Diferentes representaciones de un mismo concepto (por ejemplo, `"No"`, `"no"`, `" NO "`) hacen que los algoritmos traten estas entradas como categorías distintas, sesgando el aprendizaje. El formato dentro de cada campo debe ser estrictamente uniforme.
* **Trazabilidad e Imputación de Valores:**
  * Es una buena práctica llevar un registro explícito de qué campos contenían fallas, qué registros fueron afectados y qué valores se asignaron por defecto durante el proceso de corrección.
* **Verificación de Registros Duplicados:**
  * Filas idénticas generan un sesgo de memorización (*overfitting*) y distorsionan las métricas de evaluación del modelo.

---

#### 2. Estrategias Estándar para el Manejo de Datos Faltantes

Cuando la inspección confirma la presencia de datos faltantes, el Científico de Datos debe seleccionar de forma justificada entre tres estrategias metodológicas fundamentales:

| Estrategia | Descripción | Criterio de Selección |
| :--- | :--- | :--- |
| **1. Descartar Muestras (Filas)** | Eliminar las filas que contengan cualquier valor faltante. | Viable únicamente si la proporción de nulos es insignificante ($< 1\%$) y no destruye información crítica del negocio. |
| **2. Descartar Campos (Columnas)** | Eliminar la variable completa del conjunto de datos. | Se aplica cuando la columna tiene un porcentaje masivo de datos faltantes o no aporta valor predictivo al problema. |
| **3. Inferir Valores (Imputación)** | Reconstruir los valores ausentes basándose en la parte conocida del atributo o en relaciones lógicas con otras variables. | Es la estrategia preferida cuando existe un fundamento operativo o de negocio claro para asignar el valor sin alterar la distribución original. |

---

In [ ]:
# 1. Validación de Nulos Explícitos (NaN / None)
nulos_explicitos = churn_df.isna().sum()
total_nulos_explicitos = nulos_explicitos.sum()

print("=" * 60)
print("1. EVALUACIÓN DE VALORES NULOS EXPLÍCITOS (NaN)")
print("=" * 60)
print(f"Total de registros 'NaN' detectados por Pandas: {total_nulos_explicitos}")
if total_nulos_explicitos > 0:
    print("\nDetalle por columna:")
    print(nulos_explicitos[nulos_explicitos > 0])
else:
    print("-> No se encontraron nulos explícitos en la carga inicial.\n")


# 2. Búsqueda de Nulos Implícitos (Espacios en blanco)
columnas_object = churn_df.select_dtypes(include=['object']).columns

print("=" * 60)
print("2. EVALUACIÓN DE NULOS IMPLÍCITOS / ESPACIOS EN BLANCO (' ')")
print("=" * 60)

hallazgos_implicitos = 0
for col in columnas_object:
    # Conteo de celdas que contienen únicamente espacios en blanco
    espacios_blanco = churn_df[col].astype(str).str.isspace().sum()
    if espacios_blanco > 0:
        hallazgos_implicitos += espacios_blanco
        print(f"Atención: La columna '{col}' contiene {espacios_blanco} registros con espacios en blanco (' ').")

if hallazgos_implicitos == 0:
    print("-> No se detectaron celdas con espacios en blanco.\n")


# 3. Validación de Registros Duplicados Exactos
duplicados = churn_df.duplicated().sum()

print("\n" + "=" * 60)
print("3. EVALUACIÓN DE REGISTROS DUPLICADOS")
print("=" * 60)
print(f"Total de filas duplicadas exactas: {duplicados}\n")


# 4. Inspección de Consistencia e Invariancia en Categorías
print("=" * 60)
print("4. EVALUACIÓN DE CONSISTENCIA EN CATEGORÍAS (NIVELES ÚNICOS)")
print("=" * 60)

# Excluimos los identificadores y las variables financieras continuas para revisar el formato de texto
cols_evaluar_categorias = churn_df.select_dtypes(include=['object']).drop(columns=['customerID', 'TotalCharges'], errors='ignore')

for col in cols_evaluar_categorias.columns:
    valores = churn_df[col].unique().tolist()
    print(f"- {col} ({len(valores)} valores): {valores}")

1. EVALUACIÓN DE VALORES NULOS EXPLÍCITOS (NaN)
Total de registros 'NaN' detectados por Pandas: 0
-> No se encontraron nulos explícitos en la carga inicial.

2. EVALUACIÓN DE NULOS IMPLÍCITOS / ESPACIOS EN BLANCO (' ')
Atención: La columna 'TotalCharges' contiene 11 registros con espacios en blanco (' ').

3. EVALUACIÓN DE REGISTROS DUPLICADOS
Total de filas duplicadas exactas: 0

4. EVALUACIÓN DE CONSISTENCIA EN CATEGORÍAS (NIVELES ÚNICOS)
- gender (2 valores): ['Female', 'Male']
- Partner (2 valores): ['Yes', 'No']
- Dependents (2 valores): ['No', 'Yes']
- PhoneService (2 valores): ['No', 'Yes']
- MultipleLines (3 valores): ['No phone service', 'No', 'Yes']
- InternetService (3 valores): ['DSL', 'Fiber optic', 'No']
- OnlineSecurity (3 valores): ['No', 'Yes', 'No internet service']
- OnlineBackup (3 valores): ['Yes', 'No', 'No internet service']
- DeviceProtection (3 valores): ['No', 'Yes', 'No internet service']
- TechSupport (3 valores): ['No', 'Yes', 'No internet service']
- Strea

### Interpretación de Resultados

Estos resultados nos permite justificar nuestras decisiones de preprocesamiento:

1. **Confirmación de la "Falsa Limpieza":** Pandas reportó **0 nulos explícitos (`NaN`)**, lo cual podría llevar a una falsa sensación de que los datos están listos. Sin embargo, la inspección de nulos implícitos confirmó **11 registros con espacios en blanco (`" "`) en `TotalCharges`**.
2. **Causa Raíz:** Estos 11 espacios en blanco impiden que Pandas interprete `TotalCharges` como un número flotante, forzándolo a almacenarlo como texto (`object`).
3. **Calidad de Formato:** Se confirmó **0 duplicados** y se verificó que todas las variables cualitativas (`gender`, `Contract`, `PaymentMethod`, etc.) tienen un formato consistente sin inconsistencias de escritura.

---

### Decisión Metodológica y Justificación de Negocio

Frente a los 11 registros con espacios en blanco en `TotalCharges`, evaluamos las tres estrategias estándar:

* **¿Por qué NO descartar las muestras (Estrategia 1)?** Esos 11 registros corresponden a clientes nuevos con **0 meses de antigüedad (`tenure = 0`)**. Eliminar sus filas sesgaría el estudio del comportamiento inicial de los nuevos usuarios de MedTelecomunicaciones.
* **¿Por qué NO descartar el campo (Estrategia 2)?** `TotalCharges` representa el valor financiero acumulado, una de las características más relevantes para predecir el impacto económico del *Churn*.
* **Selección de la Estrategia 3 (Inferir Valores por Lógica de Negocio):** Si un usuario lleva $0$ meses en la compañía, su monto total facturado acumulado es estrictamente **$0.0$ USD**.

---

### Acciones a considerar

1. **Eliminar `customerID`:** No aporta información sobre el comportamiento del cliente y evitará que el modelo memorice registros (*overfitting*).
2. **Coerción de Tipo de Dato:** Convertir `TotalCharges` a tipo numérico flotante (`float64`), transformando los espacios en blanco a valores nulos reconocidos (`NaN`).
3. . **Imputación por Inferencia:** Reemplazar los valores `NaN` en `TotalCharges` con `0.0` (dado que corresponden a usuarios con `tenure = 0`).

#### ¿Qué significa `NaN`?

**`NaN`** es el acrónimo en inglés para ***Not a Number*** (No es un Número).

Es una representación estándar utilizada para indicar la **ausencia o falta de un valor válido** en un dato de tipo numérico o cuantitativo.

* **¿Por qué aparece?** Puede generarse cuando falta información en la recolección de los datos, cuando ocurre un error en el procesamiento (por ejemplo, dividir por cero) o cuando intentamos forzar la conversión a número de un campo que contenía texto o espacios en blanco (como ocurrió con `" "` en `TotalCharges`).
* **¿Por qué es crítico detectar `NaN`?** La gran mayoría de los algoritmos de Machine Learning (como las regresiones o los modelos lineales) operan mediante operaciones matriciales y algebraicas que **fallan o lanzan un error de entrenamiento** al encontrar un valor `NaN`, ya que matemáticamente no es posible operar con un valor inexistente.

En este escenario utilizaremos una **Regresión Logística** para predecir la probabilidad de *Churn*. Este algoritmo calcula internamente una combinación lineal de las características por medio de multiplicación de matrices y coeficientes:

  $$z = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \beta_p X_p$$

Si Scikit-Learn encuentra un solo valor `NaN` en la matriz $X$ al intentar calcular esta ecuación, **la operación algebraica se vuelve indefinida y el entrenamiento falla inmediatamente** lanzando un error (`ValueError: Input contains NaN`).

In [ ]:
# 1. Preservación del DataFrame original mediante una copia de trabajo
churn_prep_df = churn_df.copy()

# Eliminación de customerID
churn_prep_df = churn_prep_df.drop(columns=['customerID'])

# TotalCharges (los ' ' se convierten en NaN)
churn_prep_df['TotalCharges'] = pd.to_numeric(churn_prep_df['TotalCharges'], errors='coerce')

# Imputación (tenure = 0 -> TotalCharges = 0.0)
churn_prep_df['TotalCharges'] = churn_prep_df['TotalCharges'].fillna(0.0)

# Verificación de ejecución
print("=" * 60)
print("RESULTADOS OBTENIDOS")
print("=" * 60)
print(f"Tipo de dato corregido en 'TotalCharges': {churn_prep_df['TotalCharges'].dtype}")
print(f"Total de valores NaN en el conjunto de datos: {churn_prep_df.isna().sum().sum()}")
print(f"Estructura lista para modelado: {churn_prep_df.shape[0]} filas x {churn_prep_df.shape[1]} columnas")

RESULTADOS OBTENIDOS
Tipo de dato corregido en 'TotalCharges': float64
Total de valores NaN en el conjunto de datos: 0
Estructura lista para modelado: 7043 filas x 20 columnas


In [ ]:
# 1. Validación de Nulos Explícitos (NaN / None)
nulos_explicitos = churn_prep_df.isna().sum()
total_nulos_explicitos = nulos_explicitos.sum()

print("=" * 60)
print("1. EVALUACIÓN DE VALORES NULOS EXPLÍCITOS (NaN)")
print("=" * 60)
print(f"Total de registros 'NaN' detectados por Pandas: {total_nulos_explicitos}")
if total_nulos_explicitos > 0:
    print("\nDetalle por columna:")
    print(nulos_explicitos[nulos_explicitos > 0])
else:
    print("-> No se encontraron nulos explícitos en la carga inicial.\n")


# 2. Búsqueda de Nulos Implícitos (Espacios en blanco)
columnas_object = churn_prep_df.select_dtypes(include=['object']).columns

print("=" * 60)
print("2. EVALUACIÓN DE NULOS IMPLÍCITOS / ESPACIOS EN BLANCO (' ')")
print("=" * 60)

hallazgos_implicitos = 0
for col in columnas_object:
    # Conteo de celdas que contienen únicamente espacios en blanco
    espacios_blanco = churn_prep_df[col].astype(str).str.isspace().sum()
    if espacios_blanco > 0:
        hallazgos_implicitos += espacios_blanco
        print(f"Atención: La columna '{col}' contiene {espacios_blanco} registros con espacios en blanco (' ').")

if hallazgos_implicitos == 0:
    print("-> No se detectaron celdas con espacios en blanco.\n")


# 3. Validación de Registros Duplicados Exactos
duplicados = churn_prep_df.duplicated().sum()

print("\n" + "=" * 60)
print("3. EVALUACIÓN DE REGISTROS DUPLICADOS")
print("=" * 60)
print(f"Total de filas duplicadas exactas: {duplicados}\n")


# 4. Inspección de Consistencia e Invariancia en Categorías
print("=" * 60)
print("4. EVALUACIÓN DE CONSISTENCIA EN CATEGORÍAS (NIVELES ÚNICOS)")
print("=" * 60)

# Excluimos los identificadores y las variables financieras continuas para revisar el formato de texto
cols_evaluar_categorias = churn_prep_df.select_dtypes(include=['object']).drop(columns=['customerID', 'TotalCharges'], errors='ignore')

for col in cols_evaluar_categorias.columns:
    valores = churn_prep_df[col].unique().tolist()
    print(f"- {col} ({len(valores)} valores): {valores}")

1. EVALUACIÓN DE VALORES NULOS EXPLÍCITOS (NaN)
Total de registros 'NaN' detectados por Pandas: 0
-> No se encontraron nulos explícitos en la carga inicial.

2. EVALUACIÓN DE NULOS IMPLÍCITOS / ESPACIOS EN BLANCO (' ')
-> No se detectaron celdas con espacios en blanco.


3. EVALUACIÓN DE REGISTROS DUPLICADOS
Total de filas duplicadas exactas: 22

4. EVALUACIÓN DE CONSISTENCIA EN CATEGORÍAS (NIVELES ÚNICOS)
- gender (2 valores): ['Female', 'Male']
- Partner (2 valores): ['Yes', 'No']
- Dependents (2 valores): ['No', 'Yes']
- PhoneService (2 valores): ['No', 'Yes']
- MultipleLines (3 valores): ['No phone service', 'No', 'Yes']
- InternetService (3 valores): ['DSL', 'Fiber optic', 'No']
- OnlineSecurity (3 valores): ['No', 'Yes', 'No internet service']
- OnlineBackup (3 valores): ['Yes', 'No', 'No internet service']
- DeviceProtection (3 valores): ['No', 'Yes', 'No internet service']
- TechSupport (3 valores): ['No', 'Yes', 'No internet service']
- StreamingTV (3 valores): ['No', 'Yes', '

### Interpretación de los Resultados

Al volver a ejecutar la prueba, nos encontramos con uns sorpresa:

1. **Éxito en Nulos Explícitos e Implícitos:** Se confirma `0` valores `NaN` y `0` celdas con espacios en blanco. La corrección e imputación de `TotalCharges` fue 100% efectiva.
2. **Surgimiento de 22 Duplicados Exactos:** En la primera inspección teníamos `0` duplicados porque la columna `customerID` hacía que cada fila fuera artificialmente única, al eliminar `customerID`, salieron a la luz **22 registros idénticos**.

---

#### Análisis y Decisión sobre los Duplicados

* **Causa del Problema:** Filas con idéntica información en las 20 variables restantes pueden deberse a errores de digitación/sistema al registrar usuarios o a la coincidencia exacta de perfiles genéricos.
* **Riesgo en Machine Learning:** Mantener registros duplicados exactos distorsiona la distribución de los datos y puede causar **sesgo de memorización (*overfitting*)**, afectando la evaluación del modelo si la misma fila termina tanto en el conjunto de entrenamiento como en el de prueba (¿es esto una fuga de datos?).
* **Acción Derivada del Análisis:** Proceder a la eliminación de las 22 filas duplicadas mediante `.drop_duplicates()`, conservando únicamente la primera ocurrencia de cada registro.

In [ ]:
# 1. Conteo inicial de filas
filas_antes = churn_prep_df.shape[0]

# 2. Eliminación de duplicados exactos
churn_prep_df = churn_prep_df.drop_duplicates()

# 3. Verificación
filas_despues = churn_prep_df.shape[0]
duplicados_removidos = filas_antes - filas_despues

print("=" * 60)
print("VERIFICACIÓN DE ELIMINACIÓN DE DUPLICADOS")
print("=" * 60)
print(f"Filas iniciales: {filas_antes}")
print(f"Filas eliminadas: {duplicados_removidos}")
print(f"Filas finales preservadas: {filas_despues}")
print(f"Duplicados restantes en el DataFrame: {churn_prep_df.duplicated().sum()}")

VERIFICACIÓN DE ELIMINACIÓN DE DUPLICADOS
Filas iniciales: 7043
Filas eliminadas: 22
Filas finales preservadas: 7021
Duplicados restantes en el DataFrame: 0


### Identificación de características y variable objetivo

1. **La Matriz de Características ($X$):** Matriz de dimensiones $n \times p$ que contiene todas las variables independientes o predictoras (socio-demográficas, servicios contratados y métricas financieras).
2. **El Vector de Etiquetas ($y$):** Vector de dimensión $n \times 1$ que contiene la variable objetivo (*Churn*).

---

#### Mapeo Binario para la Regresión Logística

La Regresión Logística modela la probabilidad del evento de interés mediante la función sigmoide:

$$P(y = 1 \mid X) = \sigma(z) = \frac{1}{1 + e^{-z}}$$

Para que el algoritmo opere algebraicamente, la variable objetivo cualitativa (`'Yes'` / `'No'`) debe mapearse explícitamente a un formato numérico binario:
* **`1` (Clase Positiva / Evento de Interés):** El cliente canceló el servicio (`'Yes'`).
* **`0` (Clase Negativa):** El cliente permaneció en la compañía (`'No'`).

> Recuerda respetar la convención estándar de la comunidad de Machine Learning al utilizar la letra **$X$ mayúscula** (por ser una matriz bidimensional) y la letra **$y$ minúscula** (por ser un vector unidimensional).

### **Entendiendo la Regresión Logística para un Cliente de MedTelecomunicaciones**

---

#### **1. El Sistema de Pesos**

Durante la fase de entrenamiento (`.fit()`), el algoritmo analiza a miles de clientes pasados y aprende cuánto "pesa" cada característica:

* **Factores de Riesgo (Pesan Positivo $+w$):** Suman puntos al riesgo de Churn.
  * *Tener contrato mes a mes (`Month-to-month`):* $+1.2$ puntos de riesgo.
  * *Tener servicio de Fibra Óptica:* $+0.8$ puntos de riesgo.
  * *Pagar con transferencia electrónica:* $+0.5$ puntos de riesgo.

* **Factores de Anclaje / Retención (Pesan Negativo $-w$):** Restan puntos al riesgo de Churn.
  * *Cada año de antigüedad (`tenure`):* $-0.6$ puntos por cada año.
  * *Tener contrato a 2 años (`Two year`):* $-1.5$ puntos.
  * *Tener soporte técnico activado (`TechSupport`):* $-0.4$ puntos.

---

#### **2. Evaluando a dos Clientes Reales**

#### **Cliente de Alto Riesgo**
* Lleva solo 2 meses en la empresa (`tenure = 2`) $\rightarrow$ *Aporta poco anclaje.*
* Tiene contrato Mes a Mes y Fibra Óptica $\rightarrow$ *Acumula muchos puntos positivos de riesgo.*

**Resultado del balance:** Su puntaje total $z$ resulta en un valor **positivo muy alto**.

---

#### **Cliente Muy Fiel**
* Lleva 5 años con la empresa (`tenure = 60`) $\rightarrow$ *Restan muchísimos puntos de riesgo.*
* Tiene contrato a 2 años y soporte técnico activado $\rightarrow$ *Restan aún más puntos.*

**Resultado del balance:** Su puntaje total $z$ resulta en un valor **negativo muy alto**.

---

#### **Puntajes a Probabilidades**

La Regresión Logística toma ese puntaje y lo pasa por una curva que lo traduce a una **probabilidad de 0% a 100%**:

* **Puntaje sumamente positivo:** Se traduce en un **72.6% de probabilidad de Churn**.
* **Puntaje en cero (Cliente promedio / ambiguo):** Se traduce en un **50% de probabilidad** (incertidumbre).
* **Puntaje sumamente negativo:** Se traduce en un **0.4% de probabilidad de Churn**.

---

#### **4. El Umbral**

Por último, para decidir qué responder cuando ejecutamos `.predict()`, el algoritmo aplica una regla muy simple por defecto:

$$\text{Si Probabilidad } \ge 50\% \longrightarrow \hat{y} = 1 \text{ (El cliente se va a ir)}$$
$$\text{Si Probabilidad } < 50\% \longrightarrow \hat{y} = 0 \text{ (El cliente se va a quedar)}$$

Por eso, para **el cliente de alto riesgo** la predicción final fue `1` (Riesgo de Churn) y para **el cliente fiel** fue `0` (Cliente retenido).

In [ ]:
# Variable objetivo (y) y características (X)
X = churn_prep_df.drop(columns=['Churn'])
y_raw = churn_prep_df['Churn']

# Convertir 'Yes' -> 1, 'No' -> 0
y = y_raw.map({'Yes': 1, 'No': 0})

# Verificación de dimensiones y balance de clases
print("=" * 60)
print("CARACTERÍSTICAS Y OBJETIVO")
print("=" * 60)
print(f"Dimensiones de X (Matriz de características): {X.shape[0]} filas x {X.shape[1]} columnas")
print(f"Dimensiones de y (Vector objetivo): {y.shape[0]} registros")

print("\nDistribución absoluta de clases en 'y':")
print(y.value_counts())

print("\nDistribución relativa de clases en 'y' (Proporción):")
print(y.value_counts(normalize=True).round(3))

CARACTERÍSTICAS Y OBJETIVO
Dimensiones de X (Matriz de características): 7021 filas x 19 columnas
Dimensiones de y (Vector objetivo): 7021 registros

Distribución absoluta de clases en 'y':
Churn
0    5164
1    1857
Name: count, dtype: int64

Distribución relativa de clases en 'y' (Proporción):
Churn
0    0.736
1    0.264
Name: proportion, dtype: float64


### Características y Variable Objetivo

1. **Integridad Dimensional:** La matriz $X$ cuenta con **7,021 observaciones y 19 columnas predictoras**, habiendo depurado con éxito las filas duplicadas y la columna `customerID`.
2. **Representación Numérica:** El vector $y$ fue transformado exitosamente a valores binarios ($1$ para la fuga del cliente y $0$ para la permanencia).
3. **Diagnóstico de Balance de Clases:**
   * **Clase 0 (Permanencia):** $5,164$ registros ($73.6\%$).
   * **Clase 1 (Churn / Evento de Interés):** $1,857$ registros ($26.4\%$).

---

### ¿Hay desbalance?

En problemas de clasificación binaria, la distribución de clases se evalúa bajo la siguiente escala de referencia:

| Proporción de la Clase Minoritaria | Grado de Desbalance | Diagnóstico para Medellín Telecomunicaciones |
| :--- | :--- | :--- |
| **40% - 50%** | Balanceado | No requiere tratamiento especial. |
| **20% - 40%** | **Moderadamente Desbalanceado** | **NUESTRO CASO (26.4%):** Representa un comportamiento natural del negocio (la mayoría de los clientes no se van). |
| **1% - 20%** | Severamente Desbalanceado | Requiere técnicas avanzadas (submuestreo, sobremuestreo como SMOTE, ajuste de pesos de clase). |
| **< 1%** | Desbalance Extremo | Típico en detección de fraudes o fallas raras. |

---

En **MedTelecomunicaciones**, se esperaría que la gran mayoría de los clientes permanezcan (`Churn = 0`). Por eso, encontrar que un **26.4%** cancela el servicio (`Churn = 1`) muestra la dinámica real de dicha organización.

Considerando ese 26.4% debemos tener presente que:

1. **Uso de Partición Estratificada (`stratify=y`):**
   Al dividir los datos con `train_test_split`, debemos obligar a Scikit-Learn a mantener exactamente la proporción 73.6% / 26.4% tanto en el conjunto de entrenamiento como en el de prueba. De lo contrario, el conjunto de prueba podría quedar con muy pocos ejemplos de *Churn*.

2. **Cuidado con la "Trampa de la Exactitud" (*Accuracy Trap*):**
   Si un modelo simplista se limita a predecir siempre que "Nadie se va" (`0`), obtendría automáticamente un **73.6% de Accuracy** sin haber aprendido absolutamente nada sobre el patrón de fuga.

3. **Métricas de Evaluación Orientadas a la Clase Positiva:**
   Para evaluar nuestra Regresión Logística, nos enfocaremos en métricas como el **Recall** (¿qué porcentaje de los clientes que realmente se iban logramos detectar?) y el **F1-Score**, en lugar de fiarnos únicamente del *Accuracy*.

### Buena Práctica 6: División de Datos en Entrenamiento y Prueba (Train / Test Split)

---

#### 1. ¿Por qué se debe hacer la división antes del Preprocesamiento? (Prevención de *Data Leakage*)

Un error metodológico muy común es realizar transformaciones (como escalado o imputaciones) sobre todo el dataset antes de dividir los datos, esto causa lo que se conoce como **Fuga de datos**, lo que significa que el conjunto de entrenamiento "contamina" o aprende estadísticas pertenecientes al conjunto de prueba (como promedios o modas), generando métricas artificialmente optimistas que fallarán en producción.

**El flujo correcto debe ser estrictamente secuencial:**
1. Separar los datos en **Entrenamiento (`X_train`, `y_train`)** y **Prueba (`X_test`, `y_test`)**.
2. Ajustar (*fit*) cualquier transformador usando **únicamente los datos de entrenamiento**.
3. Aplicar la transformación (*transform*) tanto al conjunto de entrenamiento como al de prueba por separado.

---

#### 3. Control de Aleatoriedad (`random_state`)

Fijamos la semilla de aleatoriedad (`random_state=212`) para garantizar la **reproducibilidad** de las prácticas, siempre que se ejecute se obtendrá exactamente las mismas particiones.

### **Simulación del Error**

Vamos a aplicar transformaciones sobre todo el conjunto de datos ($X$) antes de realizar la partición (`train_test_split`)** y posteriormente de manera correcta.

---

#### ***Data Leakage***

Recuerda que la fuga de datos ocurre cuando el proceso de entrenamiento de un modelo tiene acceso a datos o estadísticas del conjunto de prueba (*Test*).

* **El error al transformar antes del Split:** Al codificar o escalar todo el dataset junto, los transformadores calculan parámetros globales que "contaminan" al conjunto de entrenamiento con información del futuro.
* **Consecuencia:** El modelo obtiene métricas **falsamente optimistas o infladas** durante el desarrollo.

---

Para la Regresión Logística, las variables cuantitativas de distintas magnitudes (`tenure`, `MonthlyCharges` y `TotalCharges`) deben estandarizarse para evitar que las variables con valores grandes dominen los gradientes del modelo.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, f1_score

# Identificamos variables numéricas continuas y categóricas
cols_continuas = ['tenure', 'MonthlyCharges', 'TotalCharges']
cols_cat = X.select_dtypes(include=['object']).columns.tolist()

# ==============================================================================
# Procedimiento incorrecto
# ==============================================================================
X_leak = X.copy()

# Transformación global de categóricas y numéricas ANTES de la división
ohe_g = OneHotEncoder(sparse_output=False, drop='first')
cat_g_enc = pd.DataFrame(ohe_g.fit_transform(X_leak[cols_cat]), columns=ohe_g.get_feature_names_out(cols_cat), index=X_leak.index)

scaler_g = StandardScaler()
num_g_scaled = pd.DataFrame(scaler_g.fit_transform(X_leak[cols_continuas]), columns=cols_continuas, index=X_leak.index)

X_global_prep = pd.concat([num_g_scaled, cat_g_enc], axis=1)

# División de datos
X_tr_a, X_te_a, y_tr_a, y_te_a = train_test_split(X_global_prep, y, test_size=0.20, random_state=212, stratify=y)

# Entrenamiento y evaluación
model_a = LogisticRegression(max_iter=1000, random_state=212)
model_a.fit(X_tr_a, y_tr_a)
y_pred_a = model_a.predict(X_te_a)

acc_a, rec_a, f1_a = accuracy_score(y_te_a, y_pred_a), recall_score(y_te_a, y_pred_a), f1_score(y_te_a, y_pred_a)


# ==============================================================================
# Procedimiento correcto
# ==============================================================================
# División de los datos
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=212, stratify=y)

ohe_clean = OneHotEncoder(sparse_output=False, drop='first')
ohe_clean.fit(X_train_raw[cols_cat])

# Este procedimiento es solo sobre entrenamiento
scaler_clean = StandardScaler()
scaler_clean.fit(X_train_raw[cols_continuas])

# Transformación de Train
tr_cat = pd.DataFrame(ohe_clean.transform(X_train_raw[cols_cat]), columns=ohe_clean.get_feature_names_out(cols_cat), index=X_train_raw.index)
tr_num = pd.DataFrame(scaler_clean.transform(X_train_raw[cols_continuas]), columns=cols_continuas, index=X_train_raw.index)
X_train_clean = pd.concat([tr_num, tr_cat], axis=1)

# Transformación de Test
te_cat = pd.DataFrame(ohe_clean.transform(X_test_raw[cols_cat]), columns=ohe_clean.get_feature_names_out(cols_cat), index=X_test_raw.index)
te_num = pd.DataFrame(scaler_clean.transform(X_test_raw[cols_continuas]), columns=cols_continuas, index=X_test_raw.index)
X_test_clean = pd.concat([te_num, te_cat], axis=1)

# Entrenamiento y evaluación
model_b = LogisticRegression(max_iter=1000, random_state=212)
model_b.fit(X_train_clean, y_train)
y_pred_b = model_b.predict(X_test_clean)

acc_b, rec_b, f1_b = accuracy_score(y_test, y_pred_b), recall_score(y_test, y_pred_b), f1_score(y_test, y_pred_b)


# ==============================================================================
# MÉTRICAS DE EVALUACIÓN
# ==============================================================================
df_metricas = pd.DataFrame({
    'Métrica de Evaluación': ['Accuracy (Exactitud)', 'Recall (Sensibilidad en Churn)', 'F1-Score'],
    'Modelo A (Con Fuga de datos)': [f"{acc_a:.4f}", f"{rec_a:.4f}", f"{f1_a:.4f}"],
    'Modelo B (Sin Fuga de datos)': [f"{acc_b:.4f}", f"{rec_b:.4f}", f"{f1_b:.4f}"]
})

print("=" * 80)
print("COMPARACIÓN DE MÉTRICAS")
print("=" * 80)
print(df_metricas.to_string(index=False))

COMPARACIÓN DE MÉTRICAS
         Métrica de Evaluación Modelo A (Con Fuga de datos) Modelo B (Sin Fuga de datos)
          Accuracy (Exactitud)                       0.8057                       0.8043
Recall (Sensibilidad en Churn)                       0.5511                       0.5484
                      F1-Score                       0.6003                       0.5974


#### **Las Métricas del Modelo A están "contaminadas" (Ilusión de Rendimiento)**
En el Modelo A, el $Accuracy$ ($0.8057$) y el $Recall$ ($0.5511$) son ligeramente superiores porque el modelo **hizo trampa**:
* Al calcular la media ($\mu$) y la desviación estándar ($\sigma$) usando todo el dataset, el escalador del Modelo A "ya conocía" la distribución completa de los datos de prueba ($Test$).
* Esa pequeña fuga le dio una ventaja artificial, inflando sus métricas de forma engañosa.

---

#### **El Modelo B refleja la Capacidad de Generalización Real**
Aunque las métricas del Modelo B son ligeramente menores ($Accuracy = 0.8043$, $Recall = 0.5484$), son **100% realistas**:
* Representan con exactitud cómo se comportará la Regresión Logística cuando **MedTelecomunicaciones** la ponga a evaluar clientes completamente nuevos en producción.
* Al aislar estrictamente el conjunto de prueba, garantizamos que el modelo no dependa de datos que no debería conocer.

---

### **Automatización con Pipelines y ColumnTransformer**

Hasta este punto, realizamos el preprocesamiento de forma manual: identificamos tipos de columnas, aplicamos `StandardScaler` a las numéricas y `OneHotEncoder` a las categóricas, y luego concatenamos los DataFrames.

Aunque el procedimiento fue correcto, requiere escribir mucho código repetitivo, sujeto a errores y deja la puerta abierta a que esto nos lleve a *Data Leakage*.

---

#### **¿Por qué usar Pipelines?**

1. **Garantía Anti-Fuga de datos:** Al integrar los transformadores y el estimador dentro de un solo objeto `Pipeline`, Scikit-Learn garantiza automáticamente que el método `.fit()` **solo se ejecute sobre los datos de entrenamiento**.
2. **Código Limpio y Mantenible:** Empaqueta todo el flujo de trabajo en una sola estructura. En lugar de aplicar múltiples transformaciones manuales, ejecutamos un único `.fit()` y `.predict()`.

---

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=212,
    stratify=y
)

cols_continuas = ['tenure', 'MonthlyCharges', 'TotalCharges']
cols_cat = X_train.select_dtypes(include=['object']).columns.tolist()

# Definición del ColumnTransformer (Preprocesador de columnas)
preprocesador = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), cols_continuas),
        ('cat', OneHotEncoder(sparse_output=False, drop='first'), cols_cat)
    ]
)

# Pipeline
lr_regresor = Pipeline(steps=[
    ('preprocesamiento', preprocesador),
    ('clasificador', LogisticRegression(max_iter=1000, random_state=212))
])

# Entrenamiento del Pipeline
# .fit() ejecuta el fit_transform del preprocesador SOLO en X_train
# y luego entrena la Regresión Logística con la matriz resultante.
lr_regresor.fit(X_train, y_train)

# Predicción y evaluación
# .predict() aplica automáticamente el transform aprendido en X_train sobre X_test
y_pred_pipe = lr_regresor.predict(X_test)

acc_pipe = accuracy_score(y_test, y_pred_pipe)
rec_pipe = recall_score(y_test, y_pred_pipe)
f1_pipe = f1_score(y_test, y_pred_pipe)

print("=" * 70)
print("RESULTADOS OBTENIDOS")
print("=" * 70)
print(f"Accuracy (Exactitud):             {acc_pipe:.4f}")
print(f"Recall (Sensibilidad en Churn):   {rec_pipe:.4f}")
print(f"F1-Score:                         {f1_pipe:.4f}")

RESULTADOS OBTENIDOS
Accuracy (Exactitud):             0.8043
Recall (Sensibilidad en Churn):   0.5484
F1-Score:                         0.5974


In [ ]:
# Cliente nuevo
nuevo_cliente = pd.DataFrame([{
    'tenure': 2,
    'MonthlyCharges': 85.5,
    'TotalCharges': 171.0,
    'Contract': 'Month-to-month',
    'InternetService': 'Fiber optic',
    'PaymentMethod': 'Electronic check',
    'PaperlessBilling': 'Yes',
    'gender': 'Female',
    'SeniorCitizen': 0,
    'Partner': 'No',
    'Dependents': 'No',
    'PhoneService': 'Yes',
    'MultipleLines': 'No',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'No',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'Yes',
    'StreamingMovies': 'No'
}])

# ==============================================================================
# PREDICCIÓN CON EL PIPELINE
# ==============================================================================
prediccion_pipeline = lr_regresor.predict(nuevo_cliente)
probabilidad_pipeline = lr_regresor.predict_proba(nuevo_cliente)[:, 1]

print("=" * 60)
print("PREDICCIÓN")
print("=" * 60)
print(f"Predicción Churn (1=Se va, 0=Se queda): {prediccion_pipeline[0]}")
print(f"Probabilidad de Abandono:               {probabilidad_pipeline[0]:.2%}")

PREDICCIÓN
Predicción Churn (1=Se va, 0=Se queda): 1
Probabilidad de Abandono:               74.21%


### **Pipeline**

1. **Eficiencia en Producción:** Transformamos un flujo manual complejo de múltiples pasos de escalado y codificación en una sola invocación de `.predict()` y `.predict_proba()`.
2. **Protección Contra Fuga de datos:** El pipeline aplicó la media, la desviación estándar y el mapeo de categorías aprendidos estrictamente durante el entrenamiento, sin contaminar el modelo.
---

In [ ]:
# 3 clientes nuevos
nuevos_clientes = pd.DataFrame([
    {
        'tenure': 2, 'MonthlyCharges': 85.5, 'TotalCharges': 171.0,
        'Contract': 'Month-to-month', 'InternetService': 'Fiber optic',
        'PaymentMethod': 'Electronic check', 'PaperlessBilling': 'Yes',
        'gender': 'Female', 'SeniorCitizen': 0, 'Partner': 'No',
        'Dependents': 'No', 'PhoneService': 'Yes', 'MultipleLines': 'No',
        'OnlineSecurity': 'No', 'OnlineBackup': 'No', 'DeviceProtection': 'No',
        'TechSupport': 'No', 'StreamingTV': 'Yes', 'StreamingMovies': 'No'
    },
    {
        'tenure': 48, 'MonthlyCharges': 20.0, 'TotalCharges': 960.0,
        'Contract': 'Two year', 'InternetService': 'No',
        'PaymentMethod': 'Mailed check', 'PaperlessBilling': 'No',
        'gender': 'Male', 'SeniorCitizen': 0, 'Partner': 'Yes',
        'Dependents': 'Yes', 'PhoneService': 'Yes', 'MultipleLines': 'No',
        'OnlineSecurity': 'No internet service', 'OnlineBackup': 'No internet service',
        'DeviceProtection': 'No internet service', 'TechSupport': 'No internet service',
        'StreamingTV': 'No internet service', 'StreamingMovies': 'No internet service'
    },
    {
        'tenure': 12, 'MonthlyCharges': 55.0, 'TotalCharges': 660.0,
        'Contract': 'One year', 'InternetService': 'DSL',
        'PaymentMethod': 'Bank transfer (automatic)', 'PaperlessBilling': 'Yes',
        'gender': 'Male', 'SeniorCitizen': 1, 'Partner': 'No',
        'Dependents': 'No', 'PhoneService': 'Yes', 'MultipleLines': 'Yes',
        'OnlineSecurity': 'Yes', 'OnlineBackup': 'No', 'DeviceProtection': 'Yes',
        'TechSupport': 'Yes', 'StreamingTV': 'No', 'StreamingMovies': 'No'
    }
])


# ==============================================================================
# SIN PIPELINE
# ==============================================================================
# Separar las columnas
nuevos_num = nuevos_clientes[cols_continuas]
nuevos_cat = nuevos_clientes[cols_cat]

# Transformar variables numéricas
nuevos_num_scaled = pd.DataFrame(
    scaler_clean.transform(nuevos_num),
    columns=cols_continuas,
    index=nuevos_clientes.index
)

# Transformar variables categóricas
nuevos_cat_enc = pd.DataFrame(
    ohe_clean.transform(nuevos_cat),
    columns=ohe_clean.get_feature_names_out(cols_cat),
    index=nuevos_clientes.index
)

# Reconstruir el DataFrame
nuevos_manual_completo = pd.concat([nuevos_num_scaled, nuevos_cat_enc], axis=1)

# Generar la predicción con el modelo suelto
prediccion_manual = model_b.predict(nuevos_manual_completo)
probabilidades_manual = model_b.predict_proba(nuevos_manual_completo)[:, 1]


# ==============================================================================
# CON PIPELINE
# ==============================================================================
prediccion_pipe = lr_regresor.predict(nuevos_clientes)
probabilidades_pipe = lr_regresor.predict_proba(nuevos_clientes)[:, 1]


# ==============================================================================
# TABLA COMPARATIVA DE RESULTADOS Y VERIFICACIÓN
# ==============================================================================
df_comparacion_operativa = pd.DataFrame({
    'Cliente': ['Cliente 1 (Riesgo)', 'Cliente 2 (Fiel)', 'Cliente 3 (Moderado)'],
    'Pred. Manual': prediccion_manual,
    'Prob. Manual': [f"{p:.2%}" for p in probabilidades_manual],
    'Pred. Pipeline': prediccion_pipe,
    'Prob. Pipeline': [f"{p:.2%}" for p in probabilidades_pipe]
})

print("=" * 80)
print("PREDICCIÓN DE NUEVOS CLIENTES")
print("=" * 80)
print(df_comparacion_operativa.to_string(index=False))

PREDICCIÓN DE NUEVOS CLIENTES
             Cliente  Pred. Manual Prob. Manual  Pred. Pipeline Prob. Pipeline
  Cliente 1 (Riesgo)             1       74.21%               1         74.21%
    Cliente 2 (Fiel)             0        0.44%               0          0.44%
Cliente 3 (Moderado)             0       13.25%               0         13.25%


### **[:, 1]**

Cuando ejecutamos el método `.predict_proba(X)` en una Regresión Logística binaria, Scikit-Learn **no devuelve un solo número**, sino un *array* de dos dimensiones con dos columnas:

$$\text{predict\_proba}(X) = \begin{bmatrix} P(y=0) & P(y=1) \end{bmatrix}$$

* **Columna 0 (`[:, 0]`):** La probabilidad de que el cliente **SE QUEDE** ($y=0$ / No Churn).
* **Columna 1 (`[:, 1]`):** La probabilidad de que el cliente **ABANDONE** ($y=1$ / Churn).

---
Si llamamos a `.predict_proba()` sin la indexación `[:, 1]` para nuestros 3 nuevos clientes, obtendríamos una matriz como esta:

| Cliente | Columna 0: $P(\text{No Churn})$ | Columna 1: $P(\text{Churn})$ | Suma ($P_0 + P_1$) |
| :--- | :---: | :---: | :---: |
| **Cliente 1 (Riesgo)** | `0.2738` (27.38%) | **`0.7262` (72.62%)** | `1.0` (100%) |
| **Cliente 2 (Fiel)** | `0.9956` (99.56%) | **`0.0044` (0.44%)** | `1.0` (100%) |
| **Cliente 3 (Moderado)** | `0.8544` (85.44%) | **`0.1456` (14.56%)** | `1.0` (100%) |

---

* **El primer valor (antes de la coma `:`)**: Significa *"selecciona TODAS las filas"* (es decir, los 3 clientes).
* **El segundo valor (después de la coma `1`)**: Significa *"selecciona únicamente la COLUMNA 1"* (la probabilidad del evento de interés: **Churn**).